# fenics-pipeline — End-to-End Testing Guide

This guide walks you through running the pipeline for the first time, from a fresh clone to a finished STL file. Follow each step in order. Every section tells you what you should see when things are working, and what to do if they are not.

---

## Before you begin

You will need the following on your Windows machine:

- **Docker Desktop** installed and running with the WSL2 backend enabled
- **WSL2** installed
- **Git** installed
- **VSCode** with the repository cloned and open
- A WSL2 terminal — either through VSCode's integrated terminal set to WSL2, or Windows Terminal

You do not need Python, FEniCSx, or any other software installed locally. Everything runs inside Docker.

---

## Part 1 — First-time machine setup

This section only needs to be done once per machine.

### Step 1.1 — Clone the repository

Open a WSL2 terminal and run:

```bash
git clone https://github.com/Dbartra1/fenics-pipeline.git
cd fenics-pipeline
```

### Step 1.2 — Configure WSL2 memory

FEniCSx and gmsh require more memory than WSL2 allocates by default. Open a **Windows PowerShell** terminal — not WSL2 — and run:

```powershell
powershell.exe -ExecutionPolicy Bypass -File scripts/setup_wsl.ps1
```

You should see:

```
Written to C:\Users\<you>\.wslconfig
Contents:
[wsl2]
memory=24GB
processors=10
swap=8GB

Shutting down WSL2 to apply config...
WSL2 memory now reported as: 23Gi
Done. Start your WSL2 terminal and run: make build && make up
```

If your machine has less than 24GB of RAM:

```powershell
.\scripts\setup_wsl.ps1 -MemoryGB 16 -Processors 8 -SwapGB 8
```

After this finishes **close PowerShell and reopen your WSL2 terminal**. WSL2 has restarted with the new limits.

### Step 1.3 — Verify the environment

From the WSL2 terminal in the `fenics-pipeline` directory:

```bash
make check
```

Expected output:

```
── fenics-pipeline environment check ──

  ✓ WSL2 memory: 23GB
  ✓ Docker daemon reachable
  ✓ Docker Compose: 2.x.x
  ✓ /dev/shm: 1.0GB
  ✓ .env present
       NOTEBOOK_DIR=.
  ✓ notebooks/: 9 .ipynb files found
  ✓ outputs/meshes/
  ✓ outputs/stl/
  ✓ outputs/reports/
  ✓ outputs/executed_nbs/

✅ All checks passed — run: make build && make up
```

If any line shows `✗`, follow the instruction printed next to it before continuing.

---

## Part 2 — Build the Docker image

This downloads and installs the full software stack inside Docker. It takes approximately 10–20 minutes the first time. Subsequent builds are much faster because Docker caches layers.

```bash
make build
```

It is finished when you see:

```
 => exporting to image
 => => naming to docker.io/library/fenics-pipeline:0.8.0
```

If the build fails with a network timeout, run `make build` again — it resumes from the cached layers.

---

## Part 3 — Start JupyterLab

```bash
make up
```

Expected output:

```
[+] Running 1/1
 ✔ Container fenics-pipeline  Started
JupyterLab: http://localhost:8888
```

Open `http://localhost:8888` in your browser. You should see the JupyterLab interface.

**What you will see in the file browser:** Because the repo root is now mounted at `/workspace`, the left panel will show your entire repository — `notebooks/`, `src/`, `scad/`, `scripts/`, `tests/`, `outputs/` and all root-level files. Navigate into `notebooks/` to find the pipeline notebooks.

If the browser shows a connection error, wait 10 seconds and refresh. If it still does not load:

```bash
docker logs fenics-pipeline
```

---

## Part 4 — Critical setup step: add Papermill tags to notebooks

**This must be done before running any notebook headlessly via `make run`.** Without the `parameters` tag on cell 0, Papermill cannot inject variable overrides and sweep mode will silently fail.

For each of the following notebooks, open it in JupyterLab and tag cell 0:

1. Open the notebook
2. Click on cell 0 (the parameters cell — it starts with `# Cell 0 — tagged: parameters`)
3. In the right sidebar click the **property inspector** icon (looks like a gear or settings panel) — if the sidebar is not visible go to **View → Right Sidebar**
4. Under **Cell Metadata** find the **Tags** field
5. Type `parameters` and press Enter
6. Save the notebook with Ctrl+S

Do this for:
- `notebooks/01_geometry_openscad.ipynb`
- `notebooks/02_mesh_gmsh.ipynb`
- `notebooks/03_fea_fenicsx.ipynb`
- `notebooks/04_simp_optimization.ipynb`
- `notebooks/05_stl_export.ipynb`
- `notebooks/pipeline_full.ipynb`

You only need to do this once. After saving, the tag is stored in the notebook file. Commit these changes afterwards:

```bash
git add notebooks/
git commit -m "Add Papermill parameters tags to pipeline notebooks"
git push
```

---

## Part 5 — Run the environment validation notebook

Open `notebooks/00_env_validation.ipynb` in JupyterLab. Make sure the kernel in the top-right corner reads **FEniCSx Pipeline**. If it shows something else, click it and select **FEniCSx Pipeline**.

Run all cells: **Run → Run All Cells**

The final cell should print:

```
  dolfinx              0.8.0  ✓
  gmsh                 4.12.2 ✓
  pyvista              0.43.10 ✓
  trimesh              4.4.0  ✓
  skimage              0.22.0 ✓
  papermill            2.6.0  ✓

  MPI comm size: 1
  MPI rank:      0
  OpenSCAD:      OpenSCAD version 2021.01
  PyVista headless render: OK

✅ Environment validated — safe to proceed to 01_geometry_openscad.ipynb
```

If any package shows `MISSING`, stop here and run `make build` again from WSL2, then `make up`, and retry.

---

## Part 6 — Run the pipeline interactively, one stage at a time

For each notebook below, navigate to `notebooks/` in the JupyterLab file browser, open the notebook, and click **Run → Run All Cells**. Check the expected output before moving to the next stage.

---

### Stage 1 — `notebooks/01_geometry_openscad.ipynb`

**What it does:** Reads `scad/params.json`, compiles `scad/base_part.scad` using OpenSCAD, and produces an STL of the part geometry.

**Run it and check Cell 3 output:**

```
Success:    True
Duration:   3.2s
STL path:   outputs/meshes/base_part.stl
STL size:   142.6 KB
```

**Check Cell 4 output:**

```
Vertices:  4820
Faces:     9636
Bounds:    [0.0, 100.0, 0.0, 60.0, 0.0, 20.0]

Preview saved: outputs/reports/base_part_geometry.png
```

Open `outputs/reports/base_part_geometry.png` from the JupyterLab file browser. You should see a 3D render of a rectangular bracket with four mounting holes.

**Check Cell 5 output:**

```
Handoff written: outputs/meshes/base_part_stage01.json
```

**If something goes wrong:**
- `openscad binary not found` — OpenSCAD did not install in the container. Run `make build` again.
- `Success: False` with geometry errors — check that `scad/base_part.scad` was saved correctly.
- STL size is 0 KB — a parameter in `scad/params.json` is invalid. Check that `wall_thickness` is less than half the part width.

---

### Stage 2 — `notebooks/02_mesh_gmsh.ipynb`

**What it does:** Takes the STL from Stage 1, generates a volumetric tetrahedral mesh, checks quality, and exports to XDMF for FEniCSx.

**Expected duration:** 30–90 seconds depending on mesh size.

**Check Cell 2 output:**

```
Success:          True
Duration:         45.2s
Nodes:            18432
Elements:         82156
Boundary tris:    6240
MSH:              outputs/meshes/base_part.msh
XDMF:             outputs/meshes/base_part.xdmf
```

**Check Cell 3 output:**

```
Elements:          82156
Aspect ratio:      mean=2.14  max=8.32  p95=4.21
Min dihedral:      18.4°
Inverted elements: 0
Passed:            True
```

`Inverted elements: 0` and `Passed: True` are the critical values. Do not proceed to Stage 3 if either of these is wrong.

Open `outputs/reports/base_part_mesh.png` — you should see the tetrahedral mesh on the part geometry. Open `outputs/reports/base_part_aspect_ratio.png` — the histogram bulk should be left of the orange warning line.

**If something goes wrong:**
- `No surfaces found after STL classification` — STL has geometry errors. Go back to Stage 1.
- `Inverted elements: N` where N > 0 — reduce `target_element_size` in `scad/params.json` by half and re-run.
- Meshing takes more than 5 minutes — increase `target_element_size` from `2.0` to `4.0` for a faster test run.

---

### Stage 3 — `notebooks/03_fea_fenicsx.ipynb`

**What it does:** Applies boundary conditions and solves the linear elasticity equations to compute displacement and stress.

**Expected duration:** 1–5 minutes depending on mesh size.

**Check Cell 3 output:**

```
Success:          True
Duration:         87.4s
DOFs:             55,296
Max displacement: 0.0412 mm
Von Mises max:    184.3 MPa
Von Mises mean:   22.1 MPa
```

All values should be positive. Displacement in the range 0.001–10mm for a steel part under 1000N is expected.

**Check Cell 4 output:**

```
Yield strength:   250.0 MPa
Von Mises max:    184.3 MPa
Safety factor:    1.36

⚠ Safety factor < 1.5 — marginal. Optimization will reduce it further.
```

A safety factor warning here is normal — it means the optimizer has meaningful work to do. If safety factor is below 1.0, go back to `scad/params.json` and either increase `wall_thickness` or reduce `load_magnitude_n`.

Open `outputs/reports/base_part_stress.png` — red/orange areas indicate high stress around the mounting holes and load point, blue areas are low stress candidates for material removal.

**If something goes wrong:**
- `ModuleNotFoundError: dolfinx` — restart the kernel (Kernel → Restart Kernel) and run all cells again.
- Solve hangs for more than 10 minutes — stop the cell, restart the kernel, set `USE_ITERATIVE_SOLVER = True` in Cell 0.
- Segmentation fault in the PyVista cell — `PYVISTA_OFF_SCREEN` should already be set in `docker-compose.yml`. Run `make down && make up` and retry.

---

### Stage 4 — `notebooks/04_simp_optimization.ipynb`

**What it does:** Runs the SIMP topology optimization loop, iteratively removing material from low-stress regions.

**Expected duration:** 20–60 minutes at default settings. You will see one line of output per iteration.

**What you should see during Cell 3:**

```
  Iter    1 | C=4.2831e+03 | Vol=0.400 | Δρ=0.1842 | 18.3s
  Iter    2 | C=3.1204e+03 | Vol=0.400 | Δρ=0.1421 | 17.8s
  ...
  Iter   47 | C=1.9204e+03 | Vol=0.400 | Δρ=0.0009 | 17.4s

✓ Converged at iteration 47 (Δρ=9.2e-04 < 1e-03)
```

Compliance (`C=`) should decrease and flatten. Volume fraction (`Vol=`) should stay near your target. Density change (`Δρ=`) should trend toward zero.

**Check Cell 3 final output:**

```
Success:          True
Converged:        True
Iterations:       47
Final compliance: 1.9204e+03
Final vol frac:   0.401
Duration:         834.2s (13.9 min)
```

Open `outputs/reports/base_part_density_final.png`. The right panel (thresholded at ρ>0.5) should show clear structural members connecting the load point to the mounting holes with material removed from the interior.

Open `outputs/reports/base_part_convergence.png`. The compliance curve should show a sharp drop then flatten. If it has not flattened by iteration 100, increase `MAX_ITERATIONS` to 150 in Cell 0 and re-run Cell 3 only.

**If something goes wrong:**
- `Baseline safety factor below SAFETY_FACTOR_MIN` — fix geometry or loads in Stage 1/3 first.
- Compliance never decreases — increase `PENAL` from `3.0` to `3.5` in Cell 0.
- Did not converge after 100 iterations — the result is still usable. Increase `MAX_ITERATIONS` if you want to push further.

---

### Stage 5 — `notebooks/05_stl_export.ipynb`

**What it does:** Voxelizes the density field, runs marching cubes, repairs the mesh, and exports a watertight STL.

**Check Cell 4 output:**

```
Running marching cubes at threshold 0.5...
Vertices:  48320
Triangles: 96640
Bounds:    X=[2.1, 97.9] Y=[1.8, 58.2] Z=[0.0, 20.0] mm
```

**Check Cell 5 output:**

```
After merge:  24160 verts, 96640 faces
No disconnected islands found
Applied 5 Laplacian smoothing iterations

Final: 24160 verts, 96640 faces
Watertight: True
Volume:     28441.32 mm³
```

`Watertight: True` is the critical value.

**Check Cell 6 output:**

```
STL exported:  outputs/stl/base_part_optimized.stl
File size:     4821.4 KB
Watertight:    True
Fill ratio:    23.7% (target was 40%)
```

Open `outputs/reports/base_part_before_after.png` — you should see the original solid bracket on the left and the topology-optimized version on the right with clear load-path structure.

**If something goes wrong:**
- `Marching cubes produced no geometry` — lower `DENSITY_THRESHOLD` from `0.5` to `0.4` in Cell 0.
- `Watertight: False` — increase `VOXEL_RESOLUTION` from `100` to `150` in Cell 0 and re-run from Cell 3.
- Spiky or disconnected STL — go back to Stage 4 Cell 0, increase `FILTER_RADIUS` from `6.0` to `9.0`, re-run the optimization from Cell 3.

---

## Part 7 — Run the test suite

```bash
make test
```

Expected output:

```
tests/test_mesh_quality.py::TestAspectRatio::test_equilateral_tet_aspect_ratio_near_one PASSED
tests/test_mesh_quality.py::TestAspectRatio::test_needle_tet_high_aspect_ratio PASSED
tests/test_mesh_quality.py::TestAspectRatio::test_multiple_tets PASSED
tests/test_mesh_quality.py::TestInvertedDetection::test_no_inverted_in_regular_tet PASSED
tests/test_mesh_quality.py::TestInvertedDetection::test_detects_inverted_tet PASSED
tests/test_mesh_quality.py::TestQualityReport::test_good_mesh_passes_default_thresholds PASSED
tests/test_mesh_quality.py::TestQualityReport::test_good_mesh_aspect_ratio_reasonable PASSED
tests/test_mesh_quality.py::TestQualityReport::test_thin_mesh_fails_strict_thresholds PASSED
tests/test_mesh_quality.py::TestQualityReport::test_quality_report_summary_is_string PASSED
tests/test_mesh_quality.py::TestQualityReport::test_gmsh_not_left_initialized PASSED
tests/test_fea_smoke.py::TestImportChain::test_dolfinx_imports PASSED
tests/test_fea_smoke.py::TestImportChain::test_mpi4py_functional PASSED
tests/test_fea_smoke.py::TestImportChain::test_petsc4py_imports PASSED
tests/test_fea_smoke.py::TestImportChain::test_ufl_imports PASSED
tests/test_fea_smoke.py::TestImportChain::test_src_fea_imports PASSED
tests/test_fea_smoke.py::TestImportChain::test_src_solver_imports PASSED
tests/test_fea_smoke.py::TestBoundaryConditions::test_dirichlet_bc_nonzero_dofs PASSED
tests/test_fea_smoke.py::TestBoundaryConditions::test_traction_tag_matches_load_hints PASSED
tests/test_fea_smoke.py::TestFEASolve::test_trivial_solve_completes PASSED
tests/test_fea_smoke.py::TestFEASolve::test_solve_result_physically_plausible PASSED
tests/test_fea_smoke.py::TestFEASolve::test_higher_load_gives_higher_stress PASSED

21 passed in 94.3s
```

All 21 tests should pass. The tests build their own minimal meshes internally and do not require any pipeline outputs to exist first.

---

## Part 8 — Run the full pipeline headlessly

Once all five stage notebooks have run successfully interactively, clean the outputs and run everything in one command:

```bash
make clean-outputs
make run
```

Papermill executes all five stage notebooks in sequence. The terminal will show progress as each stage completes:

```
──────────────────────────────────────────────────────────────
  Stage: 01_geometry_openscad.ipynb
  Output: base_part_01_geometry_20240101_120000.ipynb
──────────────────────────────────────────────────────────────
  ✓ Completed in 3.2s

──────────────────────────────────────────────────────────────
  Stage: 02_mesh_gmsh.ipynb
  Output: base_part_02_mesh_20240101_120003.ipynb
──────────────────────────────────────────────────────────────
  ✓ Completed in 45.2s

  ... and so on through Stage 5 ...

✅ All runs completed successfully
```

Executed notebooks with all cell outputs frozen are saved to `outputs/executed_nbs/`. Open any of them in JupyterLab to inspect what happened at each step without re-running anything.

---

## Part 9 — Generate and run test cases

Run `notebooks/00_generate_test_cases.ipynb` to produce the test case library. Navigate to `notebooks/` in JupyterLab and open it, then run all cells.

This creates a `test_cases/` directory at the repo root with 13 pre-defined parameter configurations. Each case is self-contained with its own `params.json` and a copy of `base_part.scad`.

To run a single test case through the pipeline headlessly from WSL2:

```bash
docker compose exec fenics-pipeline papermill \
    /workspace/notebooks/pipeline_full.ipynb \
    /workspace/outputs/executed_nbs/coarse_mesh_$(date +%Y%m%d_%H%M%S).ipynb \
    --kernel fenics-pipeline \
    -p PARAMS_JSON "test_cases/coarse_mesh/params.json" \
    -p PART_NAME "coarse_mesh"
```

The recommended order for first-time test case runs:

1. **`coarse_mesh`** — fastest run, confirms pipeline works end to end in minutes
2. **`baseline`** — confirms default parameters produce a good result
3. **`thin_wall`** and **`squat_wide`** — geometry edge cases
4. **`heavy_load`** — confirms safety factor warning fires in Stage 3
5. **`fine_mesh`** — most resource-intensive, leave until last

---

## Part 10 — Bringing your own geometry

**You have a STEP file (recommended):**

Open `notebooks/00_import_step.ipynb` in JupyterLab. Set `STEP_INPUT_PATH` and `PART_NAME` in Cell 0 and run all cells. The notebook reads the STEP file directly via gmsh's OpenCASCADE kernel — no STL conversion needed. After it completes, open `notebooks/03_fea_fenicsx.ipynb` and run normally from Stage 3.

**You have an STL file:**

From WSL2, run:

```bash
make import-stl STL=path/to/your/file.stl PART=your_part_name SIZE=2.0 FACE=top LOAD=1000
```

Then open `notebooks/02_mesh_gmsh.ipynb` and run normally from Stage 2.

---

## Part 11 — Collect your outputs

After a successful run, your key outputs are:

| File | Description |
|---|---|
| `outputs/stl/<part>_optimized.stl` | Final topology-optimized part, ready for slicing or CAD |
| `outputs/reports/<part>_before_after.png` | Side-by-side original vs optimized |
| `outputs/reports/<part>_stress.png` | Von Mises stress map from FEA |
| `outputs/reports/<part>_convergence.png` | SIMP optimization convergence history |
| `outputs/reports/<part>_density_final.png` | Final density field before and after thresholding |
| `outputs/executed_nbs/` | Full executed notebooks with all outputs, timestamped |

---

## Quick reference

```bash
# First-time setup
make check                    # validate environment
make build                    # build Docker image (~10-20 min)
make up                       # start JupyterLab on http://localhost:8888

# Daily use
make run                      # run full pipeline headlessly
make test                     # run test suite
make clean-outputs            # clear all generated files
make down                     # stop the container

# Debugging
make shell                    # bash prompt inside the container
docker logs fenics-pipeline   # container startup logs
make down && make up          # restart the container

# Geometry import
make import-stl STL=file.stl PART=name SIZE=2.0 FACE=top LOAD=1000
```

## Common troubleshooting

**`ModuleNotFoundError: dolfinx` from Papermill**
The kernel subprocess is not finding the dolfinx environment. Verify inside the container:
```bash
make shell
python -c "import dolfinx; print(dolfinx.__version__)"
```
If this fails, run `make build` to rebuild the image.

**`src` module not found in notebooks**
The repo root must be mounted at `/workspace`. Check your `.env` file — `NOTEBOOK_DIR` should be `.` not `./notebooks`. Restart the container with `make down && make up` after fixing.

**PyVista segfault**
Override the start command in `docker-compose.yml`:
```yaml
command: ["xvfb-run", "-a", "jupyter", "lab",
          "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root"]
```

**Marching cubes produces no geometry**
Lower `DENSITY_THRESHOLD` in Stage 5 Cell 0 from `0.5` to `0.4`.

**Spiky or disconnected STL**
Increase `FILTER_RADIUS` in Stage 4 Cell 0 by 50% and re-run from Stage 4 Cell 3.

**OOM during SIMP**
Set `USE_ITERATIVE_SOLVER = True` in Stage 3 Cell 0 and increase `target_element_size` in `scad/params.json`.

**Mesh quality failures**
Reduce `target_element_size` in `scad/params.json` or switch `ALGORITHM_3D` to `1` in Stage 2 Cell 0.